# Машинное обучение, ФКН ВШЭ

## Практическое задание 5. Решающие деревья

### Общая информация
Дата выдачи: 29.11.2024

Мягий дедлайн: 23:59 11.12.2024

Жестокий дедлайн: 23:59 13.12.2024

### О задании

Задание состоит из трёх разделов:
1. В первом разделе вы научитесь применять деревья из sklearn для задачи классификации. Вы посмотрите какие разделяющие поверхности деревья строят для различных датасетов и проанализируете их зависимость от различных гиперпараметров.
2. Во втором разделе вы попробуете реализовать свое решающее дерево для классификации и сравните его со стандартное имплементацией из sklearn.
3. В третьем разделе вы сделаете решающее дерево для регрессии, в листьях которого линейные модели.

### Оценивание и штрафы
Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимально допустимая оценка за работу — 12.5 баллов.

Сдавать задание после указанного срока сдачи нельзя. При выставлении неполного балла за задание в связи с наличием ошибок на усмотрение проверяющего предусмотрена возможность исправить работу на указанных в ответном письме условиях.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке.


### Формат сдачи
Задания сдаются через систему anytask. Посылка должна содержать:
* Ноутбук homework-practice-05-trees-Username.ipynb
* Модуль hw5code.py
* Ссылки на посылки в Яндекс.Контесте для обеих задач

В контест [https://contest.yandex.ru/contest/72492] нужно отправить файл hw5code.py с реализованными функциями и классами.

Username — ваша фамилия и имя на латинице именно в таком порядке

Для удобства проверки самостоятельно посчитайте свою максимальную оценку (исходя из набора решенных задач) и укажите ниже:

__Оценка:__

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.model_selection import GridSearchCV, ShuffleSplit, train_test_split
from sklearn.tree import DecisionTreeClassifier
from matplotlib.colors import Colormap, ListedColormap

sns.set(style="whitegrid")
sns.set_style("darkgrid")

# import warnings
# warnings.filterwarnings("ignore")

In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# This allows one to not use exessively .set_output(transform='polars') and also provides ~40% performance boost vs default
import sklearn

sklearn.set_config(transform_output="polars")

In [ ]:
layout_dict = dict(
    margin=dict(l=20, r=20, t=40, b=20),
    width=600,
    height=400,
    paper_bgcolor="LightSteelBlue",
    title_font_size=14,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
)

## reference:
# fig.update_layout(yaxis_title="trip count", yaxis_title_font_size=12)
# fig.update_layout(yaxis=dict(title=dict(text="trip count", font_size=12)))

# 1. Решающие деревья. Визуализация.

В этой части мы рассмотрим три простых двумерных датасета сделанных с помощью `make_moons`, `make_circles`, `make_classification` и посмотрим как ведет себя разделяющая поверхность в зависимости от различных гиперпараметров.

In [ ]:
from sklearn.datasets import make_moons, make_circles, make_classification

datasets = [
    make_circles(noise=0.2, factor=0.5, random_state=42),
    make_moons(noise=0.2, random_state=42),
    make_classification(
        n_classes=3,
        n_clusters_per_class=1,
        n_features=2,
        class_sep=0.8,
        random_state=3,
        n_redundant=0,
    ),
]

In [ ]:
palette = sns.color_palette(n_colors=3)
cmap = ListedColormap(palette)

In [ ]:
plt.figure(figsize=(15, 4))
for i, (x, y) in enumerate(datasets):
    plt.subplot(1, 3, i + 1)
    plt.scatter(x[:, 0], x[:, 1], c=y, cmap=cmap, alpha=0.8)

In [ ]:
fig_subplots = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[f"Dataset {i}" for i in range(3)],
    vertical_spacing=0.05,
)
figs = [
    px.scatter(
        data_frame=pl.DataFrame(datasets[i][0], schema={"x": float, "y": float}),
        x="x",
        y="y",
        color=datasets[i][1],
    )
    for i in range(3)
]
for i, fig in enumerate(figs):
    for trace in fig.data:
        fig_subplots.add_trace(trace, row=1, col=1 + i)
fig_subplots.update_layout(**layout_dict)
fig_subplots.update_layout(width=1400, coloraxis_showscale=False)

__Задание 1. (1 балл)__

Для каждого датасета обучите решающее дерево с параметрами по умолчанию, предварительно разбив выборку на обучающую и тестовую. Постройте разделящие поверхности (для этого воспользуйтесь функцией `plot_surface`, пример ниже). Посчитайте accuracy на обучающей и тестовой выборках. Сильно ли деревья переобучились?

In [ ]:
trainsets = []
testsets = []
for df in datasets:
    X_train, X_test, y_train, y_test = train_test_split(
        pl.DataFrame(df[0], schema={"x": float, "y": float}),
        df[1],
        test_size=0.2,
        random_state=231,
    )
    trainsets.append([X_train.lazy(), y_train])
    testsets.append([X_test.lazy(), y_test])

In [ ]:
def plot_surface_plt(clf, X, y):
    plot_step = 0.01
    palette = sns.color_palette(n_colors=len(np.unique(y)))
    cmap = ListedColormap(palette)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, plot_step), np.arange(y_min, y_max, plot_step)
    )
    plt.tight_layout(h_pad=0.5, w_pad=0.5, pad=2.5)

    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    cs = plt.contourf(xx, yy, Z, cmap=cmap, alpha=0.3)

    plt.scatter(
        X[:, 0],
        X[:, 1],
        c=y,
        cmap=cmap,
        alpha=0.7,
        edgecolors=np.array(palette)[y],
        linewidths=2,
    )

In [ ]:
def plot_surface(clf, X, y):
    plot_step = 0.01
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xs = np.arange(x_min, x_max, plot_step)
    ys = np.arange(y_min, y_max, plot_step)
    xx, yy = np.meshgrid(xs, ys)
    Z = clf.predict(
        pl.DataFrame(np.c_[xx.ravel(), yy.ravel()], schema={"x": float, "y": float})
    )
    Z = Z.reshape(len(ys), len(xs))

    fig = go.Figure(
        go.Contour(
            z=Z,
            x=xs,  # horizontal axis
            y=ys,  # vertical axis
            showscale=False,
            colorscale="Cividis",
            opacity=0.5,
        )
    )

    # sc = px.scatter(X, x="x", y="y", color=y, color_continuous_scale="Cividis")
    # for trace in sc.data:
    #     fig.add_trace(trace)

    fig.add_trace(
        go.Scatter(
            x=X["x"],
            y=X["y"],
            mode="markers",
            marker={
                "color": y,
                "colorscale": "Cividis",
            },
        )
    )

    fig.update_layout(**layout_dict)
    fig.update_layout()

    return fig

In [ ]:
i = 2
dtc = DecisionTreeClassifier().fit(trainsets[i][0].collect(), trainsets[i][1])
plot_surface(dtc, trainsets[i][0].collect(), trainsets[i][1])

In [ ]:
from sklearn.metrics import accuracy_score  # , precision_score, recall_score

print(
    f"Test accuracy: {accuracy_score(testsets[i][1], dtc.predict(testsets[i][0].collect())):.3f}; train Accuracy {accuracy_score(trainsets[i][1], dtc.predict(trainsets[i][0].collect())):.3f}"
)

__Задание 2. (1.25 баллов)__

Попробуйте перебрать несколько параметров для регуляризации (напр. `max_depth`, `min_samples_leaf`). Для каждого набора гиперпараметров постройте разделяющую поверхность, выведите обучающую и тестовую ошибки / accuracy. Можно делать кросс-валидацию или просто разбиение на трейн и тест, главное делайте каждый раз одинаковое разбиение, чтобы можно было корректно сравнивать (помните же, что итоговое дерево сильно зависит от небольшого изменения обучающей выборки?). Проследите как меняется разделяющая поверхность и обобщающая способность. Почему так происходит, одинаково ли изменение для разных датасетов?

__Бонус (0.75 баллов)__

Вместо того, чтобы рисовать  кучу графиков, сделайте интерактивную визуализацию разделяющей гиперплоскости с помощью библиотеки `plotly` (конкретнее, вам пригодится `plotly.graph_objects`): у вас должен получиться виджет с ползунком, по которому можно выбрать параметры `max_depth` и `min_samples_leaf` и посмотреть, как в зависимости от них меняется разделяющая поверхность и прогнозы модели. Если всё сделать аккуратно, получится очень красиво. Помните, что при загрузке в anytask виджеты могут много весить и надо подождать. Если ваш ноутбук не загружается -- попробуйте загрузить сначала с очищенным выводом этой ячейки. 

Заранее предупреждаем, что бонус сложный. Полезно будет ознакомиться:
 - https://plotly.com/python/sliders/
 - https://plotly.com/python/dropdowns/
 - https://plotly.com/python/knn-classification/

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold, cross_val_score

i = 2

max_depth_list = [2, 8, 16]
min_samples_leaf_list = [1, 4, 8, 32]

cv = KFold(n_splits=5, shuffle=True, random_state=241)
gs = GridSearchCV(
    DecisionTreeClassifier(random_state=241),
    param_grid={
        "max_depth": max_depth_list,
        "min_samples_leaf": min_samples_leaf_list,
    },
    cv=cv,
    scoring="r2",
)

gs.fit(trainsets[i][0].collect(), trainsets[i][1])
gs.best_params_

In [ ]:
from itertools import product

i = 2

figs = []
acc_test = []
acc_train = []
for ix, (maxd, mins) in enumerate(product(max_depth_list, min_samples_leaf_list)):
    dtc = DecisionTreeClassifier(max_depth=maxd, min_samples_leaf=mins)
    dtc.fit(trainsets[i][0].collect(), trainsets[i][1])

    acc_train.append(
        accuracy_score(trainsets[i][1], dtc.predict(trainsets[i][0].collect()))
    )
    acc_test.append(
        accuracy_score(testsets[i][1], dtc.predict(testsets[i][0].collect()))
    )

    fig = plot_surface(dtc, trainsets[i][0].collect(), trainsets[i][1])
    figs.append(fig)


fig_subplots = make_subplots(
    rows=len(max_depth_list),
    cols=len(min_samples_leaf_list),
    subplot_titles=[
        f"max_depth={maxd}, min_sample={mins}"
        for maxd, mins in product(max_depth_list, min_samples_leaf_list)
    ],
    vertical_spacing=0.1,
)

for ix, (fig, a_test, a_train) in enumerate(zip(figs, acc_test, acc_train)):
    for trace in fig.data:
        fig_subplots.add_trace(
            trace,
            row=1 + ix // len(min_samples_leaf_list),
            col=1 + ix % len(min_samples_leaf_list),
        )
        fig_subplots.update_xaxes(
            dict(
                title=f"Test acc: {a_test:.3f}, train acc: {a_train:.3f}",
                title_font_size=12,
            ),
            row=1 + ix // len(min_samples_leaf_list),
            col=1 + ix % len(min_samples_leaf_list),
        )

fig_subplots.update_layout(**layout_dict)
fig_subplots.update_layout(
    width=1400, height=900, coloraxis_showscale=False, showlegend=False
)

__Ответ:__

# 2. Решающие деревья своими руками

В этой части вам нужно реализовать свой класс для обучения решающего дерева в задаче бинарной классификации с возможностью обработки вещественных и категориальных признаков.

__Задание 3. (1.5 балл)__

Реализуйте функцию find_best_split из модуля hw5code.py

In [ ]:
%load_ext autoreload


In [ ]:
%autoreload 2
from hw5code_sol import *

__Задание 4. (0.5 балла)__

Загрузите таблицу [students.csv](https://github.com/esokolov/ml-course-hse/blob/master/2022-fall/homeworks-practice/homework-practice-05-trees/students.csv) (это немного преобразованный датасет [User Knowledge](https://archive.ics.uci.edu/ml/datasets/User+Knowledge+Modeling)). В ней признаки объекта записаны в первых пяти столбцах, а в последнем записана целевая переменная (класс: 0 или 1). Постройте на одном изображении пять кривых "порог — значение критерия Джини" для всех пяти признаков. Отдельно визуализируйте scatter-графики "значение признака — класс" для всех пяти признаков.

In [ ]:
df = pl.scan_csv(
    "../../../2022-fall/homeworks-practice/homework-practice-05-trees/students.csv"
)

In [ ]:
res = pl.DataFrame(schema={'threshold':float, 'gini':float, 'col':pl.String})
for col in df.collect_schema().names()[:-1]:
    tmp = find_best_split(df.collect()[col], df.collect()['UNS'])
    res = res.extend(pl.DataFrame({'threshold':tmp[0], 'gini':tmp[1], 'col':col}))

In [ ]:
fig = px.line(
    res,
    x='threshold',
    y='gini',
    color='col',
)
fig.update_layout(**layout_dict)
# fig.update_traces(mode="lines+markers")

In [ ]:
col1 = 'STG'
col2 = 'LPR'
fig=px.scatter(
    df.collect(),
    x=col1,
    y=col2,
    color='UNS',
)
fig.update_layout(**layout_dict)
fig.update_layout(coloraxis_showscale=False)

__Задание 5. (0.5 балла)__

Исходя из кривых значений критерия Джини, по какому признаку нужно производить деление выборки на два поддерева? Согласуется ли этот результат с визуальной оценкой scatter-графиков? Как бы охарактеризовали вид кривой для "хороших" признаков, по которым выборка делится почти идеально? Чем отличаются кривые для признаков, по которым деление практически невозможно?

**Ответ:**

__Задание 6. (1.5 балла).__

Разберитесь с уже написанным кодом в классе DecisionTree модуля hw5code.py. Найдите ошибки в реализации метода \_fit_node. Напишите функцию \_predict_node.

 Построение дерева осуществляется согласно базовому жадному алгоритму, предложенному в [лекции](https://github.com/esokolov/ml-course-hse/blob/master/2020-fall/lecture-notes/lecture07-trees.pdf) в разделе «Построение дерева». Выбор лучшего разбиения необходимо производить по критерию Джини. Критерий останова: все объекты в листе относятся к одному классу или ни по одному признаку нельзя разбить выборку. Ответ в листе: наиболее часто встречающийся класс в листе. Для категориальных признаков выполняется преобразование, описанное в лекции в разделе «Учет категориальных признаков».

__Задание 7. (0.5 балла)__

Протестируйте свое решающее дерево на датасете [mushrooms](https://archive.ics.uci.edu/ml/datasets/Mushroom). Вам нужно скачать таблицу agaricus-lepiota.data (лежит на гитхабе вместе с заданием), прочитать ее с помощью pandas, применить к каждому столбцу LabelEncoder (из sklearn), чтобы преобразовать строковые имена категорий в натуральные числа. Первый столбец — это целевая переменная (e — edible, p — poisonous) Мы будем измерять качество с помощью accuracy, так что нам не очень важно, что будет классом 1, а что — классом 0. Обучите решающее дерево на половине случайно выбранных объектов (признаки в датасете категориальные) и сделайте предсказания для оставшейся половины. Вычислите accuracy.

У вас должно получиться значение accuracy, равное единице (или очень близкое к единице), и не очень глубокое дерево.

In [ ]:
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ

__Задание 8. (1 балл)__

Реализуйте в классе DecisionTree поддержку параметров max_depth, min_samples_split и min_samples_leaf по аналогии с DecisionTreeClassifier. Постройте графики зависимости качества предсказания в зависимости от этих параметров для набора данных tic-tac-toe (https://github.com/esokolov/ml-course-hse/blob/master/2024-fall/homework-practice/homework-practice-05-trees/tic-tac-toe-endgame.csv).

In [ ]:
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ

__Задание 9. (до 3-х баллов)__

Реализуйте класс `LinearRegressionTree`:

 - Если вам удобно, можете сделать его наследуемым от `DecisionTree` и переопределить только необходимые методы. Можете добавить новые — как вам нравится.
 - В листьях находятся не константные предсказания, а линейные модели (можно использовать из библиотеки `sklearn`).
 - Ваша реализация должна решать задачу __регрессии__, поэтому для поиска оптимального разбиения нужно написать новую функцию.
 - **Максимум 1.8 балла, если**:
   - В качестве критерия для разбиения считаете среднее квадратное/абсолютное отклонение
   - Перебираете все пороги
   - Ваша реализация строится как обычное решающее дерево для регрессии, но в листьях линейные модели.
   - Есть поддержка параметра max_depth
 - **Максимум три балла, если выполнено следующее**:
     - Для разбиения перебираются не все пороги. Пороги выбираются из значений признаков, разбитых на квантили.
     - Для разбиении выбирается порог, который минимизирует суммарную ошибку линейных моделей после разбиения: $$\text{loss} = \frac{n_{left}}{n} \cdot \text{loss}_{left} + \frac{n_{right}}{n} \cdot \text{loss}_{right}$$ (Разумеется, для оценки этих ошибок вам надо будет строить много линейных моделей, это не дисперсии. В качестве функционала ошибки возьмите MAE или MSE)
     - Есть поддержка параметров max_depth, min_samples_split, min_samples_leaf

__Задание 10. (1 балл)__

Проведите эксперименты с реализованным вами линейным деревом на любом подходящем датасете из sklearn (https://scikit-learn.org/1.5/datasets/real_world.html), который вам нравится. Подберите лучшие гиперпараметры (max_depth и остальные, если вы их реализовывали). Сравните ваше дерево со стандартным деревом для регрессии из sklearn, для него тоже подберите гиперпараметры.

Посмотрите, что будет, если обучить ваше дерево на данных, которые сгенерированы внизу. Нарисуйте график с предсказаниями и таргетами на всей выборке, сравните с обычным деревом.

Напишите, какие достоинства и недостатки вы видите у реализованного вами линейного дерева.

In [ ]:
n_samples = 3_000
x = np.linspace(0, 5, n_samples).reshape(-1, 1)
y = np.sin(x.flatten()) + np.random.normal(0, 0.1, n_samples) * np.random.normal(
    0, 1, n_samples
)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.95, random_state=0x4B524F4C2D562D53544F594C4F % (2**32 - 1)
)

# YOUR CODE

**Ответ:**

Вставьте что угодно, описывающее ваши впечатления от этого задания: